# Cross-trait and multi-polytranscriptional risk score analysis of PD in AMP-PD: Data preparation

**Project**: Cross-trait and multi-polytranscriptomic score analysis of Parkinson's disease identifies novel associations and improves prediction

**Date last updated**: July 2026 

 # Initial set-up 

## Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os
import sys

# Bring in Pandas for Dataframe functionality
import pandas as pd
from functools import reduce

# Bring some visualization functionality 
import seaborn as sns  

# numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Enable interaction with the FireCloud API
from firecloud import api as fapi

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

## Defining shell functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))

# Utility routines for reading files from Google Cloud Storage
def gcs_read_file(path):
    """Return the contents of a file in GCS"""
    contents = !gsutil -u {BILLING_PROJECT_ID} cat {path}
    return '\n'.join(contents)
    
def gcs_read_csv(path, sep=None):
    """Return a DataFrame from the contents of a delimited file in GCS"""
    return pd.read_csv(StringIO(gcs_read_file(path)), sep=sep, engine='python')

# Utility routine for displaying a message and link to Cloud Console
def link_to_cloud_console_gcs(description, link_text, gcs_path):
    url = '{}?{}'.format(
        os.path.join('https://console.cloud.google.com/storage/browser',
                     gcs_path.replace("gs://","")),
        urllib.parse.urlencode({'userProject': BILLING_PROJECT_ID}))

    display_html_link(description, link_text, url)

## Set paths

In [ ]:
# Set up billing project and data path variables
BILLING_PROJECT_ID = os.environ['GOOGLE_PROJECT']
WORKSPACE_NAMESPACE = os.environ['WORKSPACE_NAMESPACE']
WORKSPACE_NAME = os.environ['WORKSPACE_NAME']
WORKSPACE_BUCKET = os.environ['WORKSPACE_BUCKET']
WORKSPACE_ATTRIBUTES = fapi.get_workspace(WORKSPACE_NAMESPACE, WORKSPACE_NAME).json().get('workspace',{}).get('attributes',{})

## Print the information to check we are in the proper release and billing 
## This will be different for you, the user, depending on the billing project your workspace is on
print('Billing and Workspace')
print(f'Workspace Name @ `WORKSPACE_NAME`: {WORKSPACE_NAME}')
print(f'Billing Project @ `BILLING_PROJECT_ID`: {BILLING_PROJECT_ID}')
print(f'Workspace Bucket, where you can upload and download data @ `WORKSPACE_BUCKET`: {WORKSPACE_BUCKET}')
print('')

## AMP-PD v4.0
# Explicitly define release v4.0 path 
AMP_RELEASE_CASE_CONTROL_PATH = 'gs://path/removed'
AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH = 'gs://path/removed'
AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH = 'gs://path/removed'

#rnaseq_WB-RWTS-VHBS_samples.csv
#rnaseq_WB-RWTS_samples.csv


print('AMP-PD v4.0')
print(f'Path to AMP-PD v4.0 case/control data: {AMP_RELEASE_CASE_CONTROL_PATH}')
print(f'Path to AMP-PD v4.0 PPMI and PDBP RNA Data: {AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH}')
print(f'Path to AMP-PD v4.0 HBS Data: {AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH}')

## Make working directory and install some packages for R that we will use later

NOTE: Installing these packages takes a while...

In [ ]:
# Make a general directory called "/home/jupyter/multiTRS/"
WORK_DIR = f'/home/jupyter/multiTRS/'

!mkdir -p $WORK_DIR
!echo "The working directory is: $WORK_DIR"

In [ ]:
!pip install rpy2

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install.packages(suppressMessages(c("data.table","dplyr","stringr")))
install.packages(suppressMessages("BiocManager"))
BiocManager::install("edgeR")
BiocManager::install("sva")
install.packages("tidyr")
install.packages("ggplot2")

# Copy Over Files 

## Phenotype files

### Check files exists

In [ ]:
#Check the case/control data is in the main AMP-PD release 4 release path
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {AMP_RELEASE_CASE_CONTROL_PATH}')

### Make various subdirectories

In [ ]:
# Make a subdirectory for phenotype files
WORK_DIR_PHENO = f'/home/jupyter/multiTRS/pheno'

#!mkdir -p $WORK_DIR_PHENO
!echo "The pheno directory is: $WORK_DIR_PHENO"

### Copy over the phenotype files

In [ ]:
# Copy over the case/control and demographics data etc to the working directory
print(f'The working directory is: {WORK_DIR_PHENO}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_RELEASE_CASE_CONTROL_PATH}/amp_pd_case_control.csv {WORK_DIR_PHENO}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_RELEASE_CASE_CONTROL_PATH}/clinical/Demographics.csv {WORK_DIR_PHENO}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_RELEASE_CASE_CONTROL_PATH}/clinical/Enrollment.csv {WORK_DIR_PHENO}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_RELEASE_CASE_CONTROL_PATH}/clinical/MDS_UPDRS_Part_II.csv {WORK_DIR_PHENO}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_RELEASE_CASE_CONTROL_PATH}/clinical/MDS_UPDRS_Part_III.csv {WORK_DIR_PHENO}')

### Copy over the list of EUR unrelated individuals

We may use these later for sensitivity analyses

In [ ]:
#Check the data is in the workspace bucket
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {WORKSPACE_BUCKET}')

# We can have a look at the data like this:
#shell_do(f'gsutil -u {BILLING_PROJECT_ID} cat {WORKSPACE_BUCKET}/AMPPD_EUR.COVS.txt | head')

shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {WORKSPACE_BUCKET}/AMPPD_EUR.COVS.txt {WORK_DIR_PHENO}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {WORKSPACE_BUCKET}/AMPPD_v3_COV_wPHENOS_wGENOTOOLS.csv {WORK_DIR_PHENO}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {WORKSPACE_BUCKET}/toRemove_1stand2ndDegree_Relateds_EUR.txt {WORK_DIR_PHENO}')


## Copy over the RNA files and sample inventory files

### Check files exist

In [ ]:
#Check the PPMI/PDBP data is in the RNA path
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH}')

# We can have a look at the data like this:
shell_do(f'gsutil -u {BILLING_PROJECT_ID} cat {AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH}/aggregated.genes.tsv | head')


# Check the HBS RNA data is in the RNA path
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH}')

# We can have a look at the data like this:
shell_do(f'gsutil -u {BILLING_PROJECT_ID} cat {AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH}/aggregated.genes.tsv | head')

### Make RNA subdirectories

In [ ]:
# Make subdirectory for RNA files
WORK_DIR_RNA = "/home/jupyter/multiTRS/RNA"

#!mkdir -p $WORK_DIR_RNA
!echo "The main RNA directory is: $WORK_DIR_RNA"

# and make a subdirectory for raw files before we process them further
WORK_DIR_RNA_RAW = "/home/jupyter/multiTRS/RNA/raw"

#!mkdir -p $WORK_DIR_RNA_RAW
!echo "The raw RNA directory is: $WORK_DIR_RNA_RAW"

### Copy over the sample inventory files

These will be used to QC so as to retain only baseline samples

In [ ]:
# Copy over the sample inventory to the working directory
print(f'The working directory is: {WORK_DIR_RNA_RAW}')
# First PPMI and PDBP
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_RELEASE_CASE_CONTROL_PATH}/rnaseq_WB-RWTS-VHBS_sample_inventory.csv {WORK_DIR_RNA_RAW}') 
# Then HBS
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_RELEASE_CASE_CONTROL_PATH}/rnaseq_WB-RWTS_sample_inventory.csv {WORK_DIR_RNA_RAW}')

### Copy over the RNA-seq data

In [ ]:
# Copy over the PPMI/PDBP RNA data
print(f'The working directory is: {WORK_DIR_RNA_RAW}')
# First PPMI and PDBP
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH}/aggregated.genes.tsv {WORK_DIR_RNA_RAW}')

In [ ]:
# Rename after copying to avoid clash with HBS file (which has the same name)
!ls $WORK_DIR_RNA_RAW
!mv {WORK_DIR_RNA_RAW}/aggregated.genes.tsv {WORK_DIR_RNA_RAW}/all_samples_aggregated_gene.tsv
# Check again after mv
!ls $WORK_DIR_RNA_RAW

In [ ]:
# Then HBS
print(f'The working directory is: {WORK_DIR_RNA_RAW}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -n -r {AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH}/aggregated.genes.tsv {WORK_DIR_RNA_RAW}')

In [ ]:
# Rename after copying so files are clear
!ls $WORK_DIR_RNA_RAW
!mv {WORK_DIR_RNA_RAW}/aggregated.genes.tsv {WORK_DIR_RNA_RAW}/HBS_aggregated_gene.tsv
# Check again after mv
!ls $WORK_DIR_RNA_RAW

# QC files for TRS calculation

In [ ]:
# Check these files are where they are meant to be
!ls $WORK_DIR_PHENO
!ls $WORK_DIR_RNA_RAW

## Create the case/control file with covariates/demographics

In [ ]:
%%R
# Load packages
library(data.table)
library(dplyr)

# Read in the case control and demgraphic data
case_control <- fread("/home/jupyter/multiTRS/pheno/amp_pd_case_control.csv")
print(head(case_control))
demo <- fread("/home/jupyter/multiTRS/pheno/Demographics.csv")
print(head(demo))


# Retain only cases and controls at baseline, excluding "others"
case_control <- case_control %>% filter(case_control_other_at_baseline != "Other")

tbl <- table(case_control$case_control_other_at_baseline)

print(paste(
  "After excluding others the number of cases is", tbl["Case"],
  "and the number of controls is", tbl["Control"]
))

case_control <- case_control %>% filter(case_control_other_latest != "Other")

tbl <- table(case_control$case_control_other_latest)

print(paste(
  "After excluding individuals with others at latest diagnosis the number of cases is", tbl["Case"],
  "and the number of controls is", tbl["Control"]
))


# We need to select only the three samples we will be using with RNA-seq: PPMI, PDBP and HBS
case_control$COHORT <- ifelse(grepl("LB-", case_control$participant_id), "LBD",
                        ifelse(grepl("PP-", case_control$participant_id), "PPMI",
                        ifelse(grepl("PD-", case_control$participant_id), "PDBP",
                        ifelse(grepl("HB-", case_control$participant_id), "HBS",
                        ifelse(grepl("LC-", case_control$participant_id), "LCC",
                        ifelse(grepl("BF-", case_control$participant_id), "BF",
                        ifelse(grepl("SU-", case_control$participant_id), "SU",
                        ifelse(grepl("SY-", case_control$participant_id), "SY", NA))))))))

case_control <- case_control %>% filter(COHORT %in% c("PPMI","PDBP","HBS"))

tbl <- table(case_control$case_control_other_at_baseline)

print(paste(
  "After selecting in cohort the number of cases is", tbl["Case"],
  "and the number of controls is", tbl["Control"]
))

case_control$mismatch_controls <- ifelse(case_control$case_control_other_at_baseline == "Control" & case_control$case_control_other_latest == "Case", 1, 0)

case_control <- case_control %>% filter(mismatch_controls == 0)

case_control$mismatch_cases <- ifelse(case_control$case_control_other_at_baseline == "Case" & case_control$case_control_other_latest %in% c("Control","Other"), 1, 0)

case_control <- case_control %>% filter(mismatch_cases == 0)

tbl <- table(case_control$case_control_other_at_baseline)

print(paste(
  "After filtering control that were later cases, the number of cases is", tbl["Case"],
  "and the number of controls is", tbl["Control"]
))


# We also want to do some recoding for logistic regression, coding cases as 1
case_control$case_control_other_at_baseline <- ifelse(case_control$case_control_other_at_baseline == "Case", 1,0)

# Print a table for each cohort before any further filtering
print(table(case_control$COHORT,case_control$case_control_other_at_baseline))
print(nrow(case_control)) # 4498

case_control$COHORT <- ifelse(grepl("LB-", case_control$participant_id), "LBD",
                        ifelse(grepl("PP-", case_control$participant_id), "PPMI",
                        ifelse(grepl("PD-", case_control$participant_id), "PDBP",
                        ifelse(grepl("HB-", case_control$participant_id), "HBS",
                        ifelse(grepl("LC-", case_control$participant_id), "LCC",
                        ifelse(grepl("BF-", case_control$participant_id), "BF",
                        ifelse(grepl("SU-", case_control$participant_id), "SU",
                        ifelse(grepl("SY-", case_control$participant_id), "SY", NA))))))))

# And we need to match with demographics data to use as covariates later, and do some basic qc
# First lets get the cohorts

demo$COHORT <- ifelse(grepl("LB-", demo$participant_id), "LBD",
                        ifelse(grepl("PP-", demo$participant_id), "PPMI",
                        ifelse(grepl("PD-", demo$participant_id), "PDBP",
                        ifelse(grepl("HB-", demo$participant_id), "HBS",
                        ifelse(grepl("LC-", demo$participant_id), "LCC",
                        ifelse(grepl("BF-", demo$participant_id), "BF",
                        ifelse(grepl("SU-", demo$participant_id), "SU",
                        ifelse(grepl("SY-", demo$participant_id), "SY", NA))))))))

demo <- demo %>% filter(COHORT %in% c("PPMI","PDBP","HBS"))
print(table(demo$visit_month))

# Retain only individuals measured at baseline
demo <- demo %>% filter(visit_month <= 0)

# Retain only individuals reporting as white
demo <- demo %>% filter(race == "White", ethnicity != "Hispanic or Latino")
print(demo)

# Code female as 1
demo$sex <- ifelse(demo$sex == "Female",1,0)

demo <- demo %>% filter(age_at_baseline >= 50)

# Bind the case/control and demographic data
combined <- inner_join(case_control,demo, by = c("participant_id","COHORT"))
print(head(combined))
print(table(combined$COHORT,combined$case_control_other_at_baseline))
print(table(combined$COHORT,combined$diagnosis_latest))


write.table(combined,"/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)


### Copy these files to the workspace directory to save

In [ ]:
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r {WORK_DIR_PHENO}/*w.demographics.txt {WORKSPACE_BUCKET}')

### Check they copied over

In [ ]:
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {WORKSPACE_BUCKET}')

## Filter the sample inventory files for matching with the RNA-seq data

### For PPMI and PDBP

In [ ]:
%%R

library(data.table)
library(dplyr)

case_control <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")
sample_inventory <- fread("/home/jupyter/multiTRS/RNA/raw/rnaseq_WB-RWTS_sample_inventory.csv")

# We need to select only the three samples we will be using with RNA-seq: PPMI, PDBP and HBS
sample_inventory$COHORT <- ifelse(grepl("LB-", sample_inventory$participant_id), "LBD",
                        ifelse(grepl("PP-", sample_inventory$participant_id), "PPMI",
                        ifelse(grepl("PD-", sample_inventory$participant_id), "PDBP",
                        ifelse(grepl("HB-", sample_inventory$participant_id), "HBS",
                        ifelse(grepl("LC-", sample_inventory$participant_id), "LCC",
                        ifelse(grepl("BF-", sample_inventory$participant_id), "BF",
                        ifelse(grepl("SU-", sample_inventory$participant_id), "SU",
                        ifelse(grepl("SY-", sample_inventory$participant_id), "SY", NA))))))))

sample_inventory <- sample_inventory %>%
    filter(participant_id %in% case_control$participant_id, COHORT %in% c("PPMI","PDBP"),visit_month == 0)


print(table(sample_inventory$COHORT))

sample_inventory_PPMI <- sample_inventory %>%
    filter(COHORT %in% c("PPMI"))

sample_inventory_PDBP <- sample_inventory %>%
    filter(COHORT %in% c("PDBP"))

write.table(sample_inventory,"/home/jupyter/multiTRS/RNA/raw/rnaseq_PPMI_PDBP_baseline_sample_inventory.txt", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

write.table(sample_inventory_PPMI,"/home/jupyter/multiTRS/RNA/raw/rnaseq_PPMI_only_baseline_sample_inventory.txt", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

write.table(sample_inventory_PDBP,"/home/jupyter/multiTRS/RNA/raw/rnaseq_PDBP_only_baseline_sample_inventory.txt", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

### For HBS

In [ ]:
%%R

library(data.table)
library(dplyr)

case_control <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")
sample_inventory <- fread("/home/jupyter/multiTRS/RNA/raw/rnaseq_WB-RWTS-VHBS_sample_inventory.csv")

# We need to select only the three samples we will be using with RNA-seq: PPMI, PDBP and HBS
sample_inventory$COHORT <- ifelse(grepl("LB-", sample_inventory$participant_id), "LBD",
                        ifelse(grepl("PP-", sample_inventory$participant_id), "PPMI",
                        ifelse(grepl("PD-", sample_inventory$participant_id), "PDBP",
                        ifelse(grepl("HB-", sample_inventory$participant_id), "HBS",
                        ifelse(grepl("LC-", sample_inventory$participant_id), "LCC",
                        ifelse(grepl("BF-", sample_inventory$participant_id), "BF",
                        ifelse(grepl("SU-", sample_inventory$participant_id), "SU",
                        ifelse(grepl("SY-", sample_inventory$participant_id), "SY", NA))))))))

sample_inventory <- sample_inventory %>%
    filter(participant_id %in% case_control$participant_id, COHORT %in% c("HBS"),visit_month == 0)


print(nrow(sample_inventory))

write.table(sample_inventory,"/home/jupyter/multiTRS/RNA/raw/rnaseq_HBS_baseline_sample_inventory.txt", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

In [ ]:
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/RNA/raw/*_baseline_sample_inventory.txt {WORKSPACE_BUCKET}')

**We can use these sample inventory files to filter the gene counts in the next step to contain only baseline quantification and reduce the file size when reading into R for processing**

## Create the RNA-seq files
**First we will use bash to filter down the RNA-seq files as they are currently huge and slow to read in**

### PPMI

In [ ]:
%%bash

cd /home/jupyter/multiTRS/RNA/raw/
ls /home/jupyter/multiTRS/RNA/raw/

head rnaseq_PPMI_only_baseline_sample_inventory.txt
head all_samples_aggregated_gene.tsv


# Create PPMI id file

cut -f2 rnaseq_PPMI_only_baseline_sample_inventory.txt > PPMI_baseline_sample_ids.txt

head PPMI_baseline_sample_ids.txt

# Filter the PPMI gene counts
awk -F'\t' 'FNR==NR {ids[$1]; next} FNR==1 || $2 in ids' PPMI_baseline_sample_ids.txt all_samples_aggregated_gene.tsv > PPMI_baseline_only_aggregated_gene.tsv

head PPMI_baseline_only_aggregated_gene.tsv

### PDBP

In [ ]:
%%bash

cd /home/jupyter/multiTRS/RNA/raw/
ls /home/jupyter/multiTRS/RNA/raw/

head rnaseq_PDBP_only_baseline_sample_inventory.txt
head all_samples_aggregated_gene.tsv


# Create PDBP id file

cut -f2 rnaseq_PDBP_only_baseline_sample_inventory.txt > PDBP_baseline_sample_ids.txt

head PDBP_baseline_sample_ids.txt

# Filter the HBS gene counts
awk -F'\t' 'FNR==NR {ids[$1]; next} FNR==1 || $2 in ids' PDBP_baseline_sample_ids.txt all_samples_aggregated_gene.tsv > PDBP_baseline_only_aggregated_gene.tsv

head PDBP_baseline_only_aggregated_gene.tsv

### HBS

In [ ]:
%%bash

cd /home/jupyter/multiTRS/RNA/raw/
ls /home/jupyter/multiTRS/RNA/raw/

head rnaseq_HBS_baseline_sample_inventory.txt
head HBS_aggregated_gene.tsv


# Create HBS id file

cut -f2 rnaseq_HBS_baseline_sample_inventory.txt > HBS_baseline_sample_ids.txt


head HBS_baseline_sample_ids.txt

# Filter the HBS gene counts
awk -F'\t' 'FNR==NR {ids[$1]; next} FNR==1 || $2 in ids' HBS_baseline_sample_ids.txt HBS_aggregated_gene.tsv > HBS_baseline_only_aggregated_gene.tsv


head HBS_baseline_only_aggregated_gene.tsv

## Process RNA-seq files and extract surrogate variables

**We will again do this one sample at a time**

### Create an output directory for the cleaned RNA

In [ ]:
# Make subdirectory for RNA files
CLEAN_DIR_RNA = "/home/jupyter/multiTRS/RNA/clean"

!mkdir -p $CLEAN_DIR_RNA

### Clean the RNA seq data

**We can do this in a loop and output the cleaned count matrix to the above new directory**

In [ ]:
%%R
# Load packages
library(data.table)
library(dplyr)
library(edgeR)
library(sva)
library(stringr)

setwd("/home/jupyter/multiTRS/RNA/raw/")

# Load in the case/control file for matching/filtering later
case_control_covar <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")
print(paste0("The number of rows in the full case/control data is ", nrow(case_control_covar)))

rna_files <- list.files(pattern = "_baseline_only_aggregated_gene.tsv")

# rna_files <- rna_files[1]

print(rna_files)

for (i in rna_files){
# Load in the RNA-seq data
RNA <- fread(i)
head(RNA)
    
cohort <- str_remove_all(i, "_baseline_only_aggregated_gene.tsv")

case_control_covar_filtered <- case_control_covar %>% filter(COHORT == cohort)
print(paste0("The number of individuals in ",cohort, " before further filtering is ", nrow(case_control_covar_filtered)))
    
case_control_covar_filtered <- case_control_covar_filtered %>% select(participant_id,age_at_baseline,sex) %>% filter(participant_id %in% RNA$participant_id)
print(paste0("The number of individuals in ",cohort, " after matching with RNA samples is ", nrow(case_control_covar_filtered)))

# Convert the salmon quantification file to a matrix of raw counts
RNA_essential_info <- RNA %>%
  dplyr::select(Name, participant_id, NumReads) %>%
  dplyr::mutate(participant_id = as.character(participant_id)) %>%
  tidyr::pivot_wider(
    names_from = participant_id,
    values_from = NumReads,
    values_fill = list(NumReads = 0)
  )


genes <- RNA_essential_info$Name
counts_table <- RNA_essential_info %>% select(-Name)
counts_mat <- as.matrix(counts_table)
rownames(counts_mat) <- genes

# Gene counts need to be integers for downstream
counts_mat <- round(counts_mat)

print(head(counts_mat[, 1:10], 10))
    
print(paste0("There are a total of ", nrow(counts_mat)," genes before we do any further filtering"))


# Create a DGEList object from the above raw counts matrix
dge <- DGEList(counts = counts_mat)
    
print(head(dge$samples))

# Filter low expression genes
keep <- filterByExpr(dge, group = NULL, min.count = 10, min.prop = 0.7)

dge_filtered <- dge[keep, , keep.lib.sizes = FALSE]

number_of_genes <- nrow(dge_filtered$counts)

print(paste0("There are a total of ", number_of_genes," genes after filtering low expressed genes"))

# Perform TMM normalization
dge_filtered <- calcNormFactors(dge_filtered, method = "TMM")

print(head(dge_filtered$samples))

# Retrieve normalized counts
logCPM <- cpm(dge_filtered, log = TRUE, prior.count = 1)


print(head(logCPM[, 1:10], 10))

write.table(logCPM,paste0("/home/jupyter/multiTRS/RNA/clean/rnaseq_",cohort,"_baseline_cleaned.txt"), sep = "\t", row.names = TRUE, col.names = NA, quote = FALSE)    
    
}


### Check the RNA files we have just generated exist before proceeding 

In [ ]:
!ls /home/jupyter/multiTRS/RNA/clean/

### Calculate surrogate variables and transpose RNA data for TRS calculation

We use the matrices from the previous step and the clinical/demographics data to "protect" key variables during SVA

In [ ]:
%%R

library(data.table)
library(dplyr)
library(stringr)
library(sva)
library(ggplot2)

set.seed(1)

setwd("/home/jupyter/multiTRS/RNA/clean/")

# Load in the case/control file for later
case_control_covar <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt")

rna_files <- list.files(pattern = "_baseline_cleaned.txt")

# test: rna_files <- rna_files[2]

print(rna_files)

for (i in rna_files){

cohort <- str_remove_all(i, "_baseline_cleaned.txt")
cohort <- str_remove_all(cohort, "rnaseq_")

gene_counts <- read.table(i, row.names=1, header=TRUE, check.names = FALSE)
    
#print(head(gene_counts[, 1:10],10))


# Sort the variables for sva
model_vars <- case_control_covar %>% select(participant_id,case_control_other_at_baseline,sex,age_at_baseline) %>% filter(participant_id %in% colnames(gene_counts))

print(paste0("Model variables for SVA are:"))
print(head(model_vars))
    
print(paste0("The mean age of ",i," is ", round(mean(model_vars$age_at_baseline),2)))
print(paste0("The sd age of ",i," is ", round(sd(model_vars$age_at_baseline),2)))
print(paste0("The percentage female of ", i, " is ", 
             round(100 * sum(model_vars$sex == 1)/nrow(model_vars), 2), "%"))

print(paste0("The number of cases and controls for ",cohort," is:"))
print(table(model_vars$case_control_other_at_baseline))   

model_vars <- as.data.frame(model_vars)

rownames(model_vars) <- model_vars$participant_id

model_vars$participant_id <- NULL


# models

mod = model.matrix(~case_control_other_at_baseline + age_at_baseline + sex, data=model_vars)

mod0 = model.matrix(~1,data=model_vars)

n.sv <- num.sv(as.matrix(gene_counts),mod,method="leek")

print(paste0("The number of identfied SVs in ",i," is ",n.sv))
    
if (n.sv == 0){print("No SVs identfied, creating data frame for TRS and moving to next dataset...")
               
                gene_counts_transposed <- t(gene_counts)
               
                print(head(gene_counts_transposed[, 1:10],10))
               
                # Convert to data.frame
                gene_counts_transposed <- as.data.frame(gene_counts_transposed)

                # Make partcipant_id column for later
                gene_counts_transposed$participant_id <- rownames(gene_counts_transposed)
                
                rownames(gene_counts_transposed) <- NULL
               
                gene_counts_transposed <- gene_counts_transposed %>% select(participant_id, everything())
               
                gene_counts_transposed[ , -1] <- scale(gene_counts_transposed[ , -1])
               
                #print(head(gene_counts_transposed[, 1:10],10))
               
                print(paste0("Writing gene counts for TRS for ",cohort,"..."))
               
                write.table(gene_counts_transposed,paste0("/home/jupyter/multiTRS/RNA/clean/rnaseq_",cohort,"_baseline_cleaned_transposed_scaled.txt"), sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)    
    
                
               next}

print("Running SVA...")

svobj <- sva(as.matrix(gene_counts),mod,mod0,n.sv=n.sv)

svs <- svobj$sv

colnames(svs) <- paste0("SV", seq_len(ncol(svs)))
    
print(head(svs))
    
svs_df <- as.data.frame(svs)

svs_df$participant_id <- rownames(model_vars)
    
svs_df <- svs_df %>% select(participant_id, everything())
    
print(head(svs_df))

print(paste0("Writing SVs for TRS for ",cohort,"..."))

write.table(svs_df,paste0("/home/jupyter/multiTRS/RNA/clean/",cohort,"_SVs.txt"), sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)    
    

# Get gene counts corrected for SVs (NOTE: we are not doing this and instead including SVs as covariates downstream)
#print("Correcting for SVs...")
#gene_counts_corrected <- limma::removeBatchEffect(as.matrix(gene_counts), covariates = svs, design = mod)
                                    
print("Creating data frame for TRS")
               
                gene_counts_transposed <- t(gene_counts)
               
                #print(head(gene_counts_transposed[, 1:10],10))
               
                # Convert to data.frame
                gene_counts_transposed <- as.data.frame(gene_counts_transposed)

                # Make partcipant_id column for later
                gene_counts_transposed$participant_id <- rownames(gene_counts_transposed)
                
                rownames(gene_counts_transposed) <- NULL
               
                gene_counts_transposed <- gene_counts_transposed %>% select(participant_id, everything())
    
                gene_counts_transposed[ , -1] <- scale(gene_counts_transposed[ , -1])
               
                #print(head(gene_counts_transposed[, 1:10],10))
    
                print(paste0("Writing gene counts for TRS for ",cohort,"..."))
    
                write.table(gene_counts_transposed,paste0("/home/jupyter/multiTRS/RNA/clean/rnaseq_",cohort,"_baseline_cleaned_transposed_scaled.txt"), sep = "\t", row.names = TRUE, col.names = NA, quote = FALSE) 
                print(paste0("Done processing ",cohort,"!"))

    
}



### Check the files exist and copy them to the workspace

In [ ]:
!cd /home/jupyter/multiTRS/RNA/clean/
!ls rnaseq_*_cleaned_transposed_scaled.txt
!ls *_SVs.txt
!wc -l *_baseline_cleaned_transposed_scaled.txt
!wc -l *_SVs.txt

In [ ]:
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/RNA/clean/*_baseline_cleaned_transposed_scaled.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/RNA/clean/*_SVs.txt {WORKSPACE_BUCKET}')

In [ ]:
#Check the data is in the workspace bucket
shell_do(f'gsutil -u {BILLING_PROJECT_ID} ls {WORKSPACE_BUCKET}')